In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time
from urllib.parse import urljoin
import csv
import os

In [16]:
df_hist = pd.read_csv("india_cities_lpg_fluctuations.csv")

# Fix the typo in Month
df_hist["Month"] = df_hist["Month"].str.replace("Febraury", "February")

# Strip ₹ and spaces from price columns
df_hist["Domestic Price"] = df_hist["Domestic Price"].str.replace("₹", "").str.strip().astype(float)
df_hist["Commercial Price"] = df_hist["Commercial Price"].str.replace("₹", "").str.strip().astype(float)

# Drop the change columns — not needed for analysis
df_hist = df_hist.drop(columns=["Domestic Change", "Commercial Change"])

print(df_hist.head(6))
print(df_hist.dtypes)

        City          Month  Domestic Price  Commercial Price
0   Agartala     April 2026          1073.5            2421.5
1   Agartala     March 2026          1073.5            2201.5
2   Agartala  February 2026          1013.5            2057.5
3  Ahmedabad     April 2026           920.0            2098.0
4  Ahmedabad     March 2026           920.0            1903.0
5  Ahmedabad  February 2026           860.0            1759.0
City                 object
Month                object
Domestic Price      float64
Commercial Price    float64
dtype: object


In [17]:
df_dist = pd.read_csv("cities_with_supply_distances.csv")

df_dist["Month"] = df_dist["Month"].str.replace("Febraury", "February")
df_dist["Domestic Price"] = df_dist["Domestic Price"].str.replace("₹", "").str.strip().astype(float)
df_dist["Commercial Price"] = df_dist["Commercial Price"].str.replace("₹", "").str.strip().astype(float)
df_dist = df_dist.drop(columns=["Domestic Change", "Commercial Change"])

# Keep only unique city-port-distance rows (no duplicates needed)
df_port = df_dist[["City", "Distance_to_Port_km", "Nearest_Port"]].drop_duplicates()

print(df_port)

                   City  Distance_to_Port_km               Nearest_Port
0              Agartala               386.44   Haldia Port, West Bengal
3             Ahmedabad               242.08       Kandla Port, Gujarat
6                Aizawl               513.92   Haldia Port, West Bengal
9              Amritsar              1060.60       Kandla Port, Gujarat
12            Bengaluru               298.64    Ennore Port, Tamil Nadu
15               Bhopal               662.64      Pipavav Port, Gujarat
18          Bhubaneswar               302.19   Haldia Port, West Bengal
21           Chandigarh              1074.97       Kandla Port, Gujarat
24              Chennai                19.51    Ennore Port, Tamil Nadu
27             Dehradun              1123.19       Kandla Port, Gujarat
30                  Diu                57.41      Pipavav Port, Gujarat
33            Fatehabad               898.87       Kandla Port, Gujarat
36              Gangtok               590.11   Haldia Port, West

In [18]:
df_brent = pd.read_csv("brent_crude_prices.csv")

df_brent = df_brent.rename(columns={"BZ=F": "Brent_Price_USD"})
df_brent["Date"] = pd.to_datetime(df_brent["Date"])

# Add a Month column to merge with LPG data later
df_brent["Month"] = df_brent["Date"].dt.to_period("M").astype(str)

# Monthly average (since LPG prices are monthly)
df_brent_monthly = df_brent.groupby("Month")["Brent_Price_USD"].mean().reset_index()
df_brent_monthly.columns = ["Month", "Avg_Brent_USD"]

print(df_brent_monthly)

     Month  Avg_Brent_USD
0  2025-10      65.277143
1  2025-11      63.679474
2  2025-12      61.628636
3  2026-01      64.765499
4  2026-02      69.406843
5  2026-03      99.599546
6  2026-04      99.549375


In [19]:
df_cons = pd.read_csv("cleaned_consumption_data.csv")

# Keep only LPG row
df_lpg_cons = df_cons[df_cons["Products"] == "LPG"].copy()

# Melt wide → long
df_lpg_cons = df_lpg_cons.drop(columns=["Total"])
df_lpg_cons = df_lpg_cons.melt(id_vars="Products", var_name="Month_Name", value_name="LPG_Consumption_KT")

print(df_lpg_cons)

   Products Month_Name  LPG_Consumption_KT
0       LPG      April                2545
1       LPG        May                2682
2       LPG       June                2554
3       LPG       July                2810
4       LPG     August                2833
5       LPG  September                2794
6       LPG    October                2871
7       LPG   November                2843
8       LPG   December                3067
9       LPG    January                3012
10      LPG   February                2822
11      LPG      March                2379


In [20]:
# We use a 'left' merge to keep all price history, and attach the port data where the 'City' names match.
master_df = pd.merge(df_hist, df_port, on="City", how="left")

print("After merging distances:")
print(master_df.head())

After merging distances:
        City          Month  Domestic Price  Commercial Price  \
0   Agartala     April 2026          1073.5            2421.5   
1   Agartala     March 2026          1073.5            2201.5   
2   Agartala  February 2026          1013.5            2057.5   
3  Ahmedabad     April 2026           920.0            2098.0   
4  Ahmedabad     March 2026           920.0            1903.0   

   Distance_to_Port_km              Nearest_Port  
0               386.44  Haldia Port, West Bengal  
1               386.44  Haldia Port, West Bengal  
2               386.44  Haldia Port, West Bengal  
3               242.08      Kandla Port, Gujarat  
4               242.08      Kandla Port, Gujarat  


In [21]:
# Merge based on the month columns
master_df = pd.merge(master_df, df_lpg_cons, left_on="Month", right_on="Month_Name", how="left")

# Clean up redundant columns we don't need anymore
master_df = master_df.drop(columns=["Products", "Month_Name"])

print("\nAfter merging consumption:")
print(master_df.head())


After merging consumption:
        City          Month  Domestic Price  Commercial Price  \
0   Agartala     April 2026          1073.5            2421.5   
1   Agartala     March 2026          1073.5            2201.5   
2   Agartala  February 2026          1013.5            2057.5   
3  Ahmedabad     April 2026           920.0            2098.0   
4  Ahmedabad     March 2026           920.0            1903.0   

   Distance_to_Port_km              Nearest_Port  LPG_Consumption_KT  
0               386.44  Haldia Port, West Bengal                 NaN  
1               386.44  Haldia Port, West Bengal                 NaN  
2               386.44  Haldia Port, West Bengal                 NaN  
3               242.08      Kandla Port, Gujarat                 NaN  
4               242.08      Kandla Port, Gujarat                 NaN  


In [10]:
# 1. Load the Brent crude data
df_brent = pd.read_csv("brent_crude_prices.csv")

# --- THE FIX ---
# Let's print the actual column names just so you can see what yfinance named it:
print("Original Brent columns:", df_brent.columns.tolist())

# Automatically grab the exact name of the price column (which is the 2nd column, index 1)
price_col_name = df_brent.columns[1] 
print(f"The price column is actually named: '{price_col_name}'")
# ---------------

# 2. Convert the 'Date' column to actual datetime objects
df_brent["Date"] = pd.to_datetime(df_brent["Date"])

# 3. Create a 'Month' column (e.g., extracting 'February' from '2026-02-15')
df_brent["Month"] = df_brent["Date"].dt.strftime('%B')

# 4. Group by Month and find the average using our dynamic price column name
df_brent_monthly = df_brent.groupby("Month")[price_col_name].mean().reset_index()

# Rename it so it's clean for our master dataset
df_brent_monthly.rename(columns={price_col_name: "Avg_Brent_Crude_USD"}, inplace=True)

# 5. Finally, merge this into our master_df
master_df = pd.merge(master_df, df_brent_monthly, on="Month", how="left")

# Save the final, unified dataset!
master_df.to_csv("master_lpg_analysis_dataset.csv", index=False)

print("\nFinal Master Dataset Ready!")
print(master_df.head())

Original Brent columns: ['Date', 'BZ=F']
The price column is actually named: 'BZ=F'

Final Master Dataset Ready!
        City          Month  Domestic Price  Commercial Price  \
0   Agartala     April 2026          1073.5            2421.5   
1   Agartala     March 2026          1073.5            2201.5   
2   Agartala  February 2026          1013.5            2057.5   
3  Ahmedabad     April 2026           920.0            2098.0   
4  Ahmedabad     March 2026           920.0            1903.0   

   Distance_to_Port_km              Nearest_Port  LPG_Consumption_KT  \
0               386.44  Haldia Port, West Bengal                 NaN   
1               386.44  Haldia Port, West Bengal                 NaN   
2               386.44  Haldia Port, West Bengal                 NaN   
3               242.08      Kandla Port, Gujarat                 NaN   
4               242.08      Kandla Port, Gujarat                 NaN   

   Avg_Brent_Crude_USD  
0                  NaN  
1            

In [22]:
# Remove the year and any extra spaces so "April 2026" becomes "April"
df_hist["Month"] = df_hist["Month"].str.replace(" 2026", "").str.strip()

In [23]:
# Save the final, unified DataFrame to a CSV file in your current working directory
master_df.to_csv("master_lpg_analysis_dataset.csv", index=False)

print("File successfully saved as 'master_lpg_analysis_dataset.csv'")

File successfully saved as 'master_lpg_analysis_dataset.csv'


In [25]:
import pandas as pd

#PROCESS BRENT CRUDE DATA
print("--- Loading Brent Crude ---")
df_brent = pd.read_csv("brent_crude_prices.csv")
df_brent["Date"] = pd.to_datetime(df_brent["Date"])
df_brent["Month"] = df_brent["Date"].dt.strftime('%B')

target_months = ["February", "March", "April"]
df_brent_filtered = df_brent[df_brent["Month"].isin(target_months)]

df_brent_monthly = df_brent_filtered.groupby("Month")["BZ=F"].mean().reset_index()
df_brent_monthly.rename(columns={"BZ=F": "Avg_Brent_Crude_USD"}, inplace=True)

print("Brent Averages Found:")
print(df_brent_monthly)
print("If this is empty, your brent_crude_prices.csv does not have Feb/Mar/Apr data!\n")

#PROCESS MASTER DATA & MERGE
print("--- Loading Master Dataset ---")
master_df = pd.read_csv("master_lpg_analysis_dataset.csv")

# THE FIX: Drop the old crude columns if they exist from previous failed attempts
cols_to_drop = ["Avg_Brent_Crude_USD", "Avg_Brent_Crude_USD_x", "Avg_Brent_Crude_USD_y"]
for col in cols_to_drop:
    if col in master_df.columns:
        master_df = master_df.drop(columns=[col])

# FORCE CLEAN: Remove " 2026" from the Month column just in case!
master_df["Month"] = master_df["Month"].astype(str).str.replace(" 2026", "").str.strip()

print("Master DF Months (After Cleaning):")
print(master_df["Month"].unique())
print("These must exactly match 'February', 'March', 'April'!\n")

# MERGE
final_master_df = pd.merge(master_df, df_brent_monthly, on="Month", how="left")

# Check for NaNs
missing_brent = final_master_df["Avg_Brent_Crude_USD"].isna().sum()
print(f"--- MERGE RESULTS ---")
print(f"Number of rows missing Brent Data: {missing_brent}")

# Save it
final_master_df.to_csv("master_lpg_analysis_dataset_final.csv", index=False)
print("\nSaved to 'master_lpg_analysis_dataset_final.csv'")

--- Loading Brent Crude ---
Brent Averages Found:
      Month  Avg_Brent_Crude_USD
0     April            99.549375
1  February            69.406843
2     March            99.599546
If this is empty, your brent_crude_prices.csv does not have Feb/Mar/Apr data!

--- Loading Master Dataset ---
Master DF Months (After Cleaning):
['April' 'March' 'February']
These must exactly match 'February', 'March', 'April'!

--- MERGE RESULTS ---
Number of rows missing Brent Data: 0

Saved to 'master_lpg_analysis_dataset_final.csv'


In [ ]:
import pandas as pd

# 1. Load the most recent master dataset
master_df = pd.read_csv("master_lpg_analysis_dataset_final.csv")

# 2. Drop the broken columns to start fresh
cols_to_drop = ["LPG_Consumption_KT", "LPG_Consumption_KT_x", "LPG_Consumption_KT_y", "Products", "Month_Name"]
for col in cols_to_drop:
    if col in master_df.columns:
        master_df = master_df.drop(columns=[col])

# 3. Load and prep your consumption data
df_cons = pd.read_csv("cleaned_consumption_data.csv")
df_lpg_cons = df_cons[df_cons["Products"] == "LPG"].copy()

if "Total" in df_lpg_cons.columns:
    df_lpg_cons = df_lpg_cons.drop(columns=["Total"])
df_lpg_cons = df_lpg_cons.melt(id_vars="Products", var_name="Month_Name", value_name="LPG_Consumption_KT")

df_lpg_cons["Month_Name"] = df_lpg_cons["Month_Name"].astype(str).str.strip()

# --- THE FIX ---
# Filter the consumption data so it ONLY contains February and March.
target_months = ["February", "March"]
df_lpg_cons = df_lpg_cons[df_lpg_cons["Month_Name"].isin(target_months)]
# ---------------

# 4. Merge it into the master dataframe!
final_master_df = pd.merge(master_df, df_lpg_cons, left_on="Month", right_on="Month_Name", how="left")

# 5. Clean up redundant columns
final_master_df = final_master_df.drop(columns=["Products", "Month_Name"])

# 6. Verify the results
print("--- VERIFICATION ---")
april_missing = final_master_df[final_master_df['Month'] == 'April']['LPG_Consumption_KT'].isna().sum()
other_missing = final_master_df[final_master_df['Month'] != 'April']['LPG_Consumption_KT'].isna().sum()

print(f"April rows with 'nil' consumption: {april_missing} (This is exactly what we want!)")
print(f"Feb/Mar rows with 'nil' consumption: {other_missing} (This should be 0!)")

# Save it!
final_master_df.to_csv("master_lpg_analysis_dataset_final.csv", index=False)
print("\nSuccess! Saved to 'master_lpg_analysis_dataset_final.csv'")

--- VERIFICATION ---
April rows with 'nil' consumption: 0 (This is exactly what we want!)
Feb/Mar rows with 'nil' consumption: 111 (This should be 0!)

Success! Saved to 'master_lpg_analysis_dataset_final.csv'
